# AS Betweenness Centrality, Weighted by Announced Address Space

This notebook downloads BGP routing table (RIB) snapshots in MRT format from one
or more [RIPE RIS](https://ris.ripe.net/) route collectors, merges them into a
single deduplicated route set, and computes the **betweenness centrality of
every transit AS**, where each AS path is weighted by the number of IPv4
addresses of the prefix it leads to.

The metric follows Liu, Luo, Chang and Su,
["Characterizing Inter-domain Rerouting by Betweenness Centrality after Disruptive Events"](https://rockykcc.github.io/pub/JSAC-betweenness-centrality-13.pdf)
(IEEE JSAC, 2013), which defines AS betweenness centrality over the AS paths
actually observed in BGP rather than over shortest paths in a topology graph.
The data-handling approach (MRT parsing with `pybgpkit`, per-prefix address
counting with a `pytricia` longest-prefix-match trie) mirrors the
[nids-bgp-control-plane-key](https://github.com/CAIDA/nids-bgp-control-plane-key)
reference notebook.

A final section computes **AS hegemony** (Fontugne, Shah & Aben, SIGCOMM 2017 /
PAM 2018) over the same snapshots: a per-viewpoint, trimmed-mean variant of the
same centrality idea that is robust to the vantage-point bias the pooled metric
carries.

## The metric

The paper (Eqn. 1) defines the betweenness centrality of an AS $v$ as

$$BC(v) = \frac{\sum_{u,w \in V} \sigma_{uw}(v)}{\sum_{u,w \in V} \sigma_{uw}}, \qquad u \neq w \neq v$$

where $\sigma_{uw}$ is the number of AS paths observed between $u$ and $w$, and
$\sigma_{uw}(v)$ is the number of those paths that pass **through** $v$ (as a
transit hop, not an endpoint). Because BGP is policy-routed, the paths are taken
directly from the collected routes, not recomputed as graph shortest paths.

**Adaptation to RIB snapshots.** A RIB dump contains one best route per
*(peer AS, prefix)* pair. We treat each such route as one AS path from $u$ (the
collector's peer AS, the first hop) to $w$ (the origin AS, the last hop); every
AS strictly between them is a transit hop. When several collectors are
aggregated, routes are deduplicated by *(prefix, AS path)*, so $P$ is the set of
**distinct** usable IPv4 routes across all of them — a path seen from ten peers
counts once, not ten times:

$$BC(v) = \frac{\left|\{\, p \in P : v \in \mathrm{transit}(p) \,\}\right|}{|P|}$$

**Address weighting.** The unweighted metric counts a path to a /8 and a path to
a /24 equally. To capture how much *address space* depends on an AS, we weight
each route by the size of its destination prefix:

$$BC_{\mathrm{w}}(v) = \frac{\sum_{p \in P,\ v \in \mathrm{transit}(p)} w\!\left(\mathrm{dst}(p)\right)}{\sum_{p \in P} w\!\left(\mathrm{dst}(p)\right)}$$

where $w(q)$ is the number of IPv4 addresses whose **longest matching prefix**
is $q$: the full size $2^{32-\mathrm{len}}$ of the prefix minus the addresses
covered by announced more-specifics inside it. This deduplication means every
routed address contributes its weight exactly once per distinct route, because
traffic to an address covered by a more-specific prefix follows the
more-specific route. Both scores lie in $[0, 1]$: $BC_{\mathrm{w}}(v)$ is the
fraction of routed address-space mass (address $\times$ distinct-path pairs)
whose control-plane route transits $v$.

## Setup

Install and import the required packages:
[`pybgpkit-parser`](https://github.com/bgpkit/bgpkit-parser) to parse MRT files,
[`pytricia`](https://github.com/jsommers/pytricia) for longest-prefix-match
lookups, and `pandas`/`matplotlib` for the results.

In [ ]:
%pip install -q pybgpkit-parser pytricia pandas matplotlib

import gc
import io
import time
import urllib.request
import ipaddress
from array import array
from collections import defaultdict
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import pybgpkit_parser as bgpkit
import pytricia

## Data: RIPE RIS `bview` snapshots

RIPE RIS collectors dump their full RIB every 8 hours (00:00, 08:00, 16:00 UTC)
as MRT `bview` files at
`https://data.ris.ripe.net/<collector>/<YYYY.MM>/bview.<YYYYMMDD>.<HHMM>.gz`.

Each collector sees only what its own peers send it, so a single one gives a
skewed view of the AS graph (see the caveats at the end). Listing several
collectors in `COLLECTORS` below merges their routes into one path set: more
$(u, w)$ samples, from more places, and a less biased centrality estimate. The
cost is runtime and download volume, not memory — the files are streamed one
after another, never held at once:

| Collector | Location | File size (2026-08-01) |
|---|---|---|
| `rrc00` | Amsterdam (multihop, most peers) | ~414 MB |
| `rrc01` | London, LINX | ~347 MB |
| `rrc10` | Milan, MIX | ~129 MB |
| `rrc06` | Otemachi, JPNAP | ~42 MB |

All collectors must share one `SNAPSHOT_DATE`/`SNAPSHOT_TIME`: the metric is
defined over paths observed at the same instant, and mixing timestamps would
blur the very thing the paper differences across time slots.

`["rrc00"]` is the default below; use `["rrc06"]` for a quick test run (the
whole notebook then completes in a few minutes), or e.g.
`["rrc00", "rrc06", "rrc10"]` for a broader view. Snapshots are cached under
`data/`, so re-running costs nothing after the first download.

In [ ]:
COLLECTORS = ["rrc00"]       # e.g. ["rrc00", "rrc06", "rrc10"] to merge vantage points
SNAPSHOT_DATE = "20260801"   # YYYYMMDD
SNAPSHOT_TIME = "0000"       # bviews exist at 0000, 0800, 1600 UTC

DATA_DIR = Path("data")

def rib_url(collector, date=None, time_=None):
    """URL of a RIPE RIS `bview` MRT dump."""
    date = date or SNAPSHOT_DATE
    time_ = time_ or SNAPSHOT_TIME
    return (f"https://data.ris.ripe.net/{collector}/"
            f"{date[:4]}.{date[4:6]}/bview.{date}.{time_}.gz")

def fetch_rib(collector, date=None, time_=None):
    """Download a `bview` dump into `data/` if not already cached; return its path."""
    date = date or SNAPSHOT_DATE
    time_ = time_ or SNAPSHOT_TIME
    path = DATA_DIR / f"bview.{collector}.{date}.{time_}.gz"
    if path.exists():
        print(f"using cached {path} ({path.stat().st_size / 1e6:,.0f} MB)")
        return path

    url = rib_url(collector, date, time_)
    def _report(count, block_size, total_size):
        done = count * block_size
        if done % (100 * 1024 * 1024) < block_size:
            print(f"  {done / 1e6:,.0f} MB...")

    path.parent.mkdir(parents=True, exist_ok=True)
    print(f"downloading {url}")
    t0 = time.time()
    tmp = path.with_suffix(".gz.part")
    urllib.request.urlretrieve(url, tmp, reporthook=_report)
    tmp.replace(path)  # only name it cached once it is complete
    print(f"done: {path.stat().st_size / 1e6:,.0f} MB in {time.time() - t0:.0f}s")
    return path

RIB_PATHS = [fetch_rib(c) for c in COLLECTORS]
LABEL = "+".join(COLLECTORS)
print(f"\n{len(RIB_PATHS)} snapshot(s): {LABEL} @ {SNAPSHOT_DATE} {SNAPSHOT_TIME}")

## Aggregating collectors into one route set

`iter_routes` is the single entry point both passes use to read data: give it a
list of MRT files (local paths or URLs) and it yields their announcements as one
stream, in the order the files are listed.

**What counts as a duplicate.** A route is identified by its *(prefix, AS path)*
pair — where the traffic goes and how it gets there. The same pair observed
again, at the same collector or another one, carries no new information about
the topology, so only its first occurrence is kept. What the aggregate then
holds is the **union of the distinct routes visible from all the collectors**:
adding a collector can only add paths the others did not see.

Duplicates are not rare noise. An AS path begins with the peer AS that sent it,
so identical pairs come from exactly the situations where double-counting would
be wrong: one peer router feeding two collectors (multihop `rrc00` plus its
local IXP collector) announcing its whole table twice, and two routers of the
same peer AS whose upstream paths are identical.

**Memory.** Deduplicating a stream means remembering every route already seen,
and an aggregate of several collectors runs to tens of millions of them. Two
things keep that affordable: the key is stored as a 64-bit hash rather than the
prefix and path strings, and the hashes live unboxed in an `array("q")` open
addressing table (`SeenKeys`) instead of a Python `set`, which would spend ~72
bytes an entry on int objects and slots. The result is ~16–32 bytes per route —
roughly a gigabyte at full-table, multi-collector scale — for about 20% more CPU
in the loop. The cost of the hashing is that two distinct routes whose keys
collide drop one of the two: below $10^{-3}$ probability across $10^8$ routes.

In [ ]:
class SeenKeys:
    """Open-addressed set of 64-bit keys, 8 bytes per slot.

    A Python `set` of the same hashes costs ~72 bytes per entry (the int objects
    plus the table); this holds the keys unboxed in an `array("q")` and linear
    probes, for 13-27 bytes per key depending on where the power-of-two table
    size lands — about a third of the `set`, for roughly 20% more CPU.

    The table doubles when it passes `MAX_LOAD`, and `expect` presizes it so a
    run whose size is known in advance never rehashes. Beyond ~0.6, linear
    probing degrades quickly (an unsuccessful lookup averages
    `(1 + 1/(1-load)**2)/2` probes: 2.5 at 0.5, 3.6 at 0.6, 8.5 at 0.75), which
    is why the table is kept below that rather than filled.

    Key 0 marks an empty slot, so a key that hashes to 0 is stored as 1 — one
    key in 2**64 is thereby merged with another, which is far below the
    collision rate of the 64-bit hash itself.
    """

    MAX_LOAD = 0.6

    def __init__(self, expect=0):
        bits = 22
        while (1 << bits) * self.MAX_LOAD < expect:
            bits += 1
        self._size_table(bits)
        self.size = 0

    def _size_table(self, bits):
        self.bits = bits
        self.mask = (1 << bits) - 1
        self.limit = int((1 << bits) * self.MAX_LOAD)
        # Repeating a one-element array builds the buffer directly; passing
        # `bytes(8 << bits)` would materialize a second full-size buffer for the
        # constructor to copy, doubling peak memory at gigabyte table sizes.
        self.table = array("q", [0]) * (1 << bits)

    def add(self, key):
        """Insert `key`; return True if it was new, False if already present."""
        if key == 0:
            key = 1
        t, mask = self.table, self.mask
        i = key & mask
        while True:
            cur = t[i]
            if cur == 0:
                t[i] = key
                self.size += 1
                if self.size > self.limit:
                    self._grow()
                return True
            if cur == key:
                return False
            i = (i + 1) & mask

    def _grow(self):
        old = self.table
        self._size_table(self.bits + 1)
        t, mask = self.table, self.mask
        for key in old:
            if key:
                i = key & mask
                while t[i]:
                    i = (i + 1) & mask
                t[i] = key

    def __len__(self):
        return self.size


def iter_routes(paths, dedup=True, expect_routes=0, progress_every=5_000_000, stats=None):
    """Stream several MRT RIB files as one aggregated, deduplicated route set.

    `paths` is an iterable of MRT files — local paths or URLs, anything
    `pybgpkit` can open. They are read one after another and yielded as a single
    stream of announcement elems (`elem_type == "A"`; withdrawals, which a
    `bview` dump does not contain anyway, are skipped). Only the deduplication
    keys are held in memory, so adding a collector costs time and bandwidth
    rather than RAM.

    A route is identified by its *(prefix, AS path)* pair. The same pair seen
    again — at this collector or another — says nothing new about the topology
    and is dropped, so what comes out is the union of the distinct routes across
    all the files. Prepending is not collapsed for the key, so the rare peer that
    prepends towards one collector and not another leaves both variants in.

    Keys are stored as 64-bit hashes in a `SeenKeys` table; the collision risk
    (one route dropped, chance under 1e-3 at 1e8 routes) buys an order of
    magnitude in memory over keeping the strings. `expect_routes` presizes that
    table — pass pass 1's announcement count, an upper bound on the number of
    distinct routes, to avoid rehashing. Pass `dedup=False` to yield every
    announcement as it appears.

    `stats`, if given, is a dict updated on exhaustion with the number of
    announcements read, routes yielded, and duplicates dropped.
    """
    seen = SeenKeys(expect_routes) if dedup else None
    n_read = n_kept = n_dupe = 0
    for path in paths:
        name = Path(str(path)).name
        for elem in bgpkit.Parser(url=str(path)):
            if elem.elem_type != "A":
                continue
            n_read += 1
            if dedup and not seen.add(hash((elem.prefix, elem.as_path))):
                n_dupe += 1
                continue
            n_kept += 1
            if progress_every and n_kept % progress_every == 0:
                print(f"  {name}: {n_kept:,} routes kept, {n_dupe:,} duplicates...")
            yield elem
    if stats is not None:
        stats.update(announcements=n_read, routes=n_kept, duplicates=n_dupe)

## Pass 1 — collect the announced IPv4 prefixes

The address weight of a prefix depends on which *other* prefixes are announced
inside it, so we need the complete set of announced prefixes — across every
snapshot being aggregated — before any path can be weighted. This first pass
only collects that set (IPv6 prefixes are excluded — address counts across the
two families are not comparable). Deduplication is switched off here: the result
is a set of prefixes, so repeated routes collapse on their own and there is no
reason to pay for the route keys. The path-counting pass comes after the weights
are built.

The pass also tallies IPv4 announcements per collector peer — a few thousand
counters that let the hegemony section at the end of the notebook pick its
full-feed viewpoints without an extra pass over the files.

In [ ]:
t0 = time.time()
raw_prefixes = set()
peer_counts = defaultdict(int)  # (peer IP, peer AS) -> IPv4 announcements, for the hegemony pass
n_entries = 0
# dedup=False: a set of prefixes is already deduplicated, and skipping the route
# keys here keeps this pass free of the memory the aggregated pass 2 needs.
for elem in iter_routes(RIB_PATHS, dedup=False, progress_every=0):
    n_entries += 1
    pfx = elem.prefix
    if ":" not in pfx:  # keep IPv4 only
        raw_prefixes.add(pfx)
        peer_counts[elem.peer_ip, elem.peer_asn] += 1
    if n_entries % 5000000 == 0:
        print(f"  {n_entries:,} entries, {len(raw_prefixes):,} IPv4 prefixes...")

print(f"{n_entries:,} RIB entries across {len(RIB_PATHS)} snapshot(s), "
      f"{len(raw_prefixes):,} unique IPv4 prefixes, "
      f"{len(peer_counts):,} collector peers ({time.time() - t0:.0f}s)")

## Address weight per prefix (longest-prefix-match deduplication)

A prefix of length $\ell$ spans $2^{32-\ell}$ addresses, but if a more-specific
prefix is announced inside it, traffic to those addresses follows the
more-specific route. Counting both at full size would double-count the nested
space. So, as in the reference notebook, each prefix is weighted by the
addresses for which it is the **longest match**: its full size minus the sizes
of the announced prefixes directly nested inside it. Subtracting only *direct*
children (prefixes whose immediate parent in the trie is this prefix) removes
each covered address exactly once, however deep the nesting goes.

Two consequences worth noting:

- A prefix completely covered by its more-specifics gets weight 0 — its paths
  still count toward the unweighted metric, but carry no address mass.
- A default route (`0.0.0.0/0`) would soak up all unannounced space, so it is
  dropped entirely (announcing a default to a collector is junk, not
  reachability).

`address_weights` below builds the trie and returns the per-prefix weights twice
over: keyed by the raw prefix strings as they appear in the MRT file (so pass 2
can look them up without re-parsing) and keyed by the normalized prefix (for
totals). The total of all weights should come out close to the routed IPv4
address space (~3.1 billion addresses, ~72% of the 2³² total).

In [ ]:
def address_weights(prefixes):
    """Address weight of every announced IPv4 prefix, deduplicated by longest match.

    `prefixes` is an iterable of prefix strings as they appear in the MRT file.
    Each prefix is credited with 2**(32 - len) addresses minus the sizes of the
    announced prefixes *directly* nested inside it, which removes each covered
    address exactly once however deep the nesting goes. IPv6 prefixes, default
    routes and unparseable prefixes are dropped.

    Returns `(weights, norm_weight)`:

    - `weights` is keyed by the raw prefix strings passed in, so a second pass
      over the MRT file can look weights up without re-parsing prefixes;
    - `norm_weight` is keyed by the normalized prefix, one entry per prefix, for
      totals and per-prefix statistics.
    """
    pyt = pytricia.PyTricia(32)
    raw_to_norm = {}
    for pfx in prefixes:
        if ":" in pfx:  # keep IPv4 only
            continue
        try:
            net = ipaddress.ip_network(pfx, strict=False)
        except ValueError:
            continue
        if net.prefixlen == 0:  # drop default routes
            continue
        norm = str(net)
        raw_to_norm[pfx] = norm
        pyt[norm] = True

    norm_weight = {}
    for pfx in pyt:
        w = 1 << (32 - int(pfx.split("/")[1]))
        for child in pyt.children(pfx):
            if child != pfx and pyt.parent(child) == pfx:
                w -= 1 << (32 - int(child.split("/")[1]))
        norm_weight[pfx] = w

    return {raw: norm_weight[norm] for raw, norm in raw_to_norm.items()}, norm_weight


In [ ]:
t0 = time.time()
weights, norm_weight = address_weights(raw_prefixes)

total_routed = sum(norm_weight.values())
zero_w = sum(1 for w in norm_weight.values() if w == 0)
print(f"{len(norm_weight):,} prefixes weighted ({time.time() - t0:.0f}s)")
print(f"  total routed IPv4 space: {total_routed:,} addresses "
      f"({100 * total_routed / 2**32:.1f}% of 2^32)")
print(f"  fully covered by more-specifics (weight 0): {zero_w:,} prefixes")


## Pass 2 — accumulate transit counts per AS

Now iterate over the aggregated route set again — this time with deduplication
on, so a *(prefix, AS path)* pair seen at several collectors is counted once —
and credit each transit AS on its AS path. Per route:

- **AS-path prepending** (the same AS repeated consecutively to make a path less
  attractive) is collapsed — it is a traffic-engineering artifact, not extra hops.
- Paths containing an **AS-set** (`{...}`, from route aggregation) are skipped;
  the actual sequence of ASes is ambiguous. These are rare (well under 0.1%).
- The first hop (the collector's peer AS, $u$) and the last hop (the origin AS,
  $w$) are endpoints, not transit — only the ASes strictly between them are
  credited, and at most once per path even if a path loops.
- Every usable route counts in the denominator, including peer-to-origin routes
  with no transit hop at all.

Every AS observed on a usable path — endpoints included — is recorded in
`all_ases`, so the ranking below covers non-transit ASes too. By the definition
of the metric, an AS that never appears in transit position (a pure edge AS)
scores 0 on both variants; it still gets a row.

**Performance note:** the lookup structures built above hold millions of
long-lived Python objects. The allocations inside this loop keep triggering
full garbage collections that re-traverse all of them, slowing the loop by
~80×. `gc.freeze()` moves the existing heap out of the collector's scope
(measured on `rrc06`: 2848s → 33s). The deduplication set is built *during* the
loop, so freezing cannot cover it and the collector is switched off outright for
the duration; nothing here creates reference cycles, so only the (unused)
cycle detector is lost, not refcounting.

In [ ]:
gc.collect()
gc.freeze()
gc.disable()   # the dedup table is built inside the loop, so freeze() cannot cover it

t0 = time.time()
transit_w = defaultdict(int)   # AS -> address-weighted path mass
transit_u = defaultdict(int)   # AS -> path count
all_ases = set()               # every AS observed on a usable path, endpoints included
total_weight = 0
total_paths = 0
n_asset = 0
route_stats = {}
# n_entries from pass 1 presizes the dedup table, so it never has to rehash.
for elem in iter_routes(RIB_PATHS, dedup=True, expect_routes=n_entries, stats=route_stats):
    w = weights.get(elem.prefix)
    if w is None:  # IPv6, default route, or unparseable prefix
        continue
    path_str = elem.as_path
    if path_str is None:
        continue
    if "{" in path_str:  # AS-set: ambiguous, skip
        n_asset += 1
        continue
    path = []
    prev = None
    for hop in path_str.split():
        if hop != prev:  # collapse prepending
            path.append(hop)
            prev = hop
    total_paths += 1
    total_weight += w
    all_ases.update(path)
    for asn in set(path[1:-1]):
        transit_u[asn] += 1
        transit_w[asn] += w

gc.enable()
gc.unfreeze()
print(f"{route_stats['announcements']:,} announcements read, "
      f"{route_stats['duplicates']:,} duplicate routes dropped "
      f"({100 * route_stats['duplicates'] / max(route_stats['announcements'], 1):.1f}%)")
print(f"{total_paths:,} distinct IPv4 paths counted ({n_asset:,} skipped for AS-sets), "
      f"{len(all_ases):,} ASes observed, {len(transit_u):,} of them transit "
      f"({time.time() - t0:.0f}s)")

## Normalize and rank

Divide by the totals to get $BC(v)$ and $BC_{\mathrm{w}}(v)$, attach AS names
and countries from RIPE's [asn.txt](https://ftp.ripe.net/ripe/asnames/asn.txt),
and rank. The table covers **every AS observed in the route set**: ASes that
never appear as a transit hop (pure edge ASes — stubs and collector peers that
transit nothing) carry $BC = 0$ by definition. The `paths` and `addresses`
columns are the raw numerators: how many routes transit the AS, and how much
address $\times$ vantage mass they carry.

In [ ]:
ASNAMES_URL = "https://ftp.ripe.net/ripe/asnames/asn.txt"
asn_info = {}
try:
    with urllib.request.urlopen(ASNAMES_URL, timeout=60) as resp:
        for line in io.TextIOWrapper(resp, encoding="utf-8", errors="replace"):
            num, _, rest = line.strip().partition(" ")
            if not num.isdigit():
                continue
            name, sep, country = rest.rpartition(", ")
            asn_info[int(num)] = (name if sep else rest, country if sep else "")
except Exception as exc:
    print(f"could not fetch AS names ({exc}); continuing with numbers only")

In [ ]:

rows = []
for asn_s in all_ases:
    wmass = transit_w.get(asn_s, 0)
    n_paths = transit_u.get(asn_s, 0)
    asn = int(asn_s)
    name, country = asn_info.get(asn, ("", ""))
    rows.append({
        "asn": asn,
        "name": name[:48],
        "country": country,
        "bc_weighted": wmass / total_weight,
        "bc_unweighted": n_paths / total_paths,
        "paths": n_paths,
        "addresses": wmass,
    })

df = (pd.DataFrame(rows)
        .sort_values("bc_weighted", ascending=False)
        .reset_index(drop=True))
df.insert(0, "rank_w", df.index + 1)
df["rank_u"] = df["bc_unweighted"].rank(ascending=False, method="min").astype(int)

df.head(25).style.format({
    "bc_weighted": "{:.4f}",
    "bc_unweighted": "{:.4f}",
    "paths": "{:,}",
    "addresses": "{:,}",
}).hide(axis="index")

## Distribution of centrality across ASes

The CCDF below shows, for each centrality value $x$, how many ASes have
$BC \geq x$. Edge ASes — the large majority, with $BC = 0$ — sit off the
log-log axes; what is plotted is the transit tail. Like the prefix- and
address-count distributions in the reference notebook, expect a heavy tail:
most transit ASes carry a tiny fraction of paths, while a handful of tier-1 and
large tier-2 networks each transit a sizable share of the routed address space.

In [ ]:
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"
BLUE, ORANGE = "#2a78d6", "#eb6834"

def style_axes(ax):
    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values():
        spine.set_color(MUTED)
        spine.set_linewidth(0.8)
    ax.tick_params(colors=MUTED, labelcolor=INK2)
    ax.grid(True, which="both", color=GRID, linewidth=0.6, alpha=0.6)
    ax.set_axisbelow(True)

def ccdf(values):
    xs = sorted(v for v in values if v > 0)  # edge ASes (BC = 0) cannot render on log axes
    return xs, [len(xs) - i for i in range(len(xs))]

fig, ax = plt.subplots(figsize=(7, 4.5))
fig.patch.set_facecolor(SURFACE)
style_axes(ax)

xw, yw = ccdf(df["bc_weighted"])
xu, yu = ccdf(df["bc_unweighted"])
ax.plot(xw, yw, color=BLUE, linewidth=2, label="address-weighted")
ax.plot(xu, yu, color=ORANGE, linewidth=2, label="unweighted (path count)")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("AS betweenness centrality", color=INK2)
ax.set_ylabel("ASes with BC ≥ x", color=INK2)
ax.set_title(f"CCDF of AS betweenness centrality ({LABEL}, {SNAPSHOT_DATE})",
             color=INK)
ax.legend(frameon=False, labelcolor=INK2)
fig.tight_layout()
plt.show()

## What the address weighting changes

Each point below is one transit AS. On the diagonal, weighting does not matter:
the AS transits an address-typical mix of prefixes. Above the diagonal, the AS
carries routes to *larger-than-average* prefixes (e.g. carriers in front of
legacy /8s and big cloud or telco aggregates) — the weighted metric promotes
it. Below the diagonal, its paths mostly lead to small prefixes (long /24-heavy
deaggregation), so its path count overstates the address space that depends on
it. The furthest-off-diagonal ASes are where this metric tells a different
story than simple path counting.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
fig.patch.set_facecolor(SURFACE)
style_axes(ax)

transit = df[(df["bc_unweighted"] > 0) & (df["bc_weighted"] > 0)]  # edge ASes sit at (0, 0)
lo = max(transit["bc_unweighted"].min(), transit["bc_weighted"].min(), 1e-8)
hi = max(transit["bc_unweighted"].max(), transit["bc_weighted"].max()) * 2
ax.plot([lo, hi], [lo, hi], color=MUTED, linewidth=1, linestyle="--", zorder=1)
ax.annotate("equal under both metrics", xy=(hi, hi),
            xytext=(-8, -14), textcoords="offset points",
            ha="right", fontsize=8, color=MUTED)

ax.scatter(transit["bc_unweighted"], transit["bc_weighted"],
           s=14, color=BLUE, alpha=0.45, linewidths=0, zorder=2)

for _, row in transit.head(8).iterrows():
    ax.annotate(f"AS{row['asn']}", xy=(row["bc_unweighted"], row["bc_weighted"]),
                xytext=(5, 3), textcoords="offset points",
                fontsize=8, color=INK2)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel("Unweighted betweenness centrality (path count)", color=INK2)
ax.set_ylabel("Address-weighted betweenness centrality", color=INK2)
ax.set_title(f"Address weighting vs. path counting ({LABEL}, {SNAPSHOT_DATE})",
             color=INK)
fig.tight_layout()
plt.show()

## Interpretation and caveats

- **Vantage-point bias.** The paths are those visible from the peers of the
  collectors in `COLLECTORS`, so ASes topologically close to them are
  over-represented. A run on `rrc06` (Tokyo) alone, for example, ranks Japanese
  carriers (KDDI AS2516, IIJ AS2497, ARTERIA AS2518) far higher than a global
  view would. Aggregating collectors is the lever against this: each one adds
  the paths its peers see and nothing else, since duplicates are dropped. It
  narrows the bias but does not remove it — RIS peers are concentrated in Europe
  and among networks willing to peer with a collector.
- **Distinct paths, not path samples.** Deduplicating by *(prefix, AS path)*
  means a path counts once no matter how many peers report it. This is a
  deliberate departure from the paper's $\sigma_{uw}$: it stops a well-connected
  peer AS from inflating the paths it happens to observe, but it also flattens
  the genuine popularity of a route. An AS on a path that 200 peers use scores
  the same as one on a path a single peer sees.
- **Peers are endpoints.** A collector peer appears as the first hop $u$ of its
  own routes and never earns transit credit from them — large ASes that peer
  with the collectors are structurally *under*-counted.
- **Control plane, not traffic.** A path weighted by addresses says how much
  address space *would* traverse the AS if every address mattered equally; real
  traffic per address varies over orders of magnitude.
- **One time slot.** The paper's actual object of study is the *variance*
  $\widetilde{BC}_t(v) = BC_t(v) - BC_{t-1}(v)$ across a sequence of time slots,
  which localizes disruptive events. Re-running this notebook over consecutive
  `bview`/update intervals — with the same `COLLECTORS`, so the vantage set is
  held fixed — and differencing the per-AS scores is the natural extension.
- **No bogon filtering.** Announcements of reserved or unallocated space are
  counted like any other; a stricter version would drop them before weighting.

## AS hegemony

The first caveat above — vantage-point bias — is structural: pooling every
observed path into one set makes ASes near the collectors' peers look more
central than they are, and adding collectors only dilutes the bias. **AS
hegemony** (Fontugne, Shah & Aben,
[*AS Hegemony: A Robust Metric for AS Centrality*](https://doi.org/10.1145/3123878.3131982),
SIGCOMM Posters 2017; extended in
[*The (thin) Bridges of AS Connectivity*](https://arxiv.org/abs/1711.02805),
PAM 2018) attacks the bias directly instead of diluting it: compute a
centrality score **per viewpoint**, then aggregate with a trimmed mean, so that
viewpoints abnormally close to (or far from) an AS get no vote on its score.

A **viewpoint** is a single full-feed BGP peer of a collector — one router,
identified here by its peer IP. For viewpoint $j$ with route set $P_j$, the
paper's (address-weighted) per-viewpoint centrality of AS $v$ is

$$BC^{(j)}(v) = \frac{\sum_{p \in P_j,\ v \in \mathrm{ases}(p)} w\!\left(\mathrm{dst}(p)\right)}{\sum_{p \in P_j} w\!\left(\mathrm{dst}(p)\right)}$$

— the fraction of $j$'s routed address space whose best path contains $v$,
with $w$ the same longest-match address weights as above. Two deliberate
differences from the $BC$ sections:

- $\mathrm{ases}(p)$ is **every distinct AS on the path, endpoints included** —
  the peer AS and the origin count. (The paper's toy example scores a stub AS
  at $1/12$ from a remote viewpoint: the path *to* it counts.) A viewpoint
  sitting inside or next to $v$ therefore reports $BC^{(j)}(v) \approx 1$ — and
  it is the trimming below, not endpoint exclusion, that removes that bias.
  Origin ASes end up with small non-zero scores — their own address mass —
  rather than the structural zero the transit-only $BC$ assigns them.
- Routes are **not pooled across viewpoints**: each viewpoint is normalized by
  its own table, so a peer feeding two collectors, or simply announcing more
  paths, cannot outvote the others.

Sorting the per-viewpoint scores $BC^{(1)}(v) \le \dots \le BC^{(n)}(v)$ and
dropping the top and bottom $\lfloor\alpha n\rfloor$ — the viewpoints biased
towards and against $v$ — the hegemony of $v$ is the mean of what remains:

$$H(v, \alpha) = \frac{1}{n - 2\lfloor\alpha n\rfloor} \sum_{j=\lfloor\alpha n\rfloor + 1}^{n - \lfloor\alpha n\rfloor} BC^{(j)}(v)$$

The paper uses $\alpha = 0.1$ and shows the scores stabilize once roughly 20
viewpoints are available, where pooled BC keeps drifting as viewpoints are
added. $H(v)$ reads as: *the fraction of a typical viewpoint's
(address-weighted) routes that pass through $v$.*

This is the paper's **global graph** — all origins together, one score per AS,
the same quantity the IHR `/hegemony/` API publishes under `originasn=0` (see
`ihr_hegemony_top10.ipynb` for a live pull of those values); its per-origin
**local graphs** are the natural next extension. Both the address-weighted
$H_w$ (the paper's choice for IPv4) and the unweighted $H_u$ (every route
counts 1) are computed below, reusing the pass-1 prefix weights unchanged.

In [ ]:
ALPHA = 0.1                # fraction of viewpoints trimmed at each end (paper's value)
FULL_FEED_FRACTION = 0.75  # a viewpoint must carry >= this fraction of all announced IPv4 prefixes

vp_threshold = FULL_FEED_FRACTION * len(norm_weight)

# peer_counts (from pass 1) sums announcements across collectors, so a peer
# feeding two collectors is counted about twice here. That only overshoots,
# never undershoots: no genuine full feed is lost at this stage, and impostors
# are re-checked against their deduplicated route count after pass 3.
vp_meta = sorted(k for k, c in peer_counts.items() if c >= vp_threshold)
vp_index = {peer_ip: j for j, (peer_ip, _) in enumerate(vp_meta)}
m = len(vp_meta)

print(f"{m} candidate full-feed viewpoints "
      f"(>= {vp_threshold:,.0f} of {len(norm_weight):,} IPv4 prefixes) "
      f"in {len({asn for _, asn in vp_meta})} peer ASes, "
      f"out of {len(peer_counts)} peers total")

### Pass 3 — per-viewpoint path mass

Hegemony needs the data in a different shape than the pooled metric: *who
reported a path* is the unit of normalization, and that is exactly what the
*(prefix, AS path)* deduplication of pass 2 erases. So the files are streamed a
third time with `iter_routes(dedup=False)`, and this pass keeps its own route
identity: **(viewpoint, prefix)** — one best route per peer per prefix. Within
a single RIB dump that pair is already unique, so the dedup table is built only
when several collectors are merged, where the same peer router can feed two of
them. (A peer duplicated in full would inflate its numerators and denominator
by the same factor and cancel out; partial session overlaps would not, so first
occurrence wins.)

Routes from peers that failed the full-feed prefilter are skipped before
anything is spent on them. Per usable route — weight known, path present, no
AS-set — **every distinct AS on the path** is credited under the reporting
viewpoint (`set()` on the hops collapses prepending on its own, and endpoints
are deliberately kept, per the metric), and the viewpoint's totals grow by the
route's weight and by one path.

The accumulators are two `array("q")` rows per AS — one 64-bit slot per
candidate viewpoint for address mass, one for path count — so memory is
$\approx 16m$ bytes per AS: a few hundred MB at full-table scale with a couple
hundred viewpoints, far below what a dict-per-viewpoint would cost. The same
`gc.freeze()` / `gc.disable()` bracket as pass 2 applies, and for the same
reason: the accumulators are built inside the loop.

In [ ]:
gc.collect()
gc.freeze()
gc.disable()   # the accumulators (and dedup table) are built inside the loop

t0 = time.time()
zero_row = array("q", [0]) * m
hege_acc = {}                      # AS -> (address mass per viewpoint, path count per viewpoint)
vp_total_w = array("q", zero_row)  # denominator per viewpoint: address mass
vp_total_u = array("q", zero_row)  # denominator per viewpoint: usable routes
# Within one RIB dump each (peer, prefix) already appears exactly once; the
# dedup table is only needed when collectors are merged and a peer feeds several.
seen = (SeenKeys(sum(c for (ip, _), c in peer_counts.items() if ip in vp_index))
        if len(RIB_PATHS) > 1 else None)
n_used = 0
for elem in iter_routes(RIB_PATHS, dedup=False, progress_every=0):
    j = vp_index.get(elem.peer_ip)
    if j is None:  # not a full-feed candidate
        continue
    if seen is not None and not seen.add(hash((elem.peer_ip, elem.prefix))):
        continue
    w = weights.get(elem.prefix)
    if w is None:  # IPv6, default route, or unparseable prefix
        continue
    path_str = elem.as_path
    if path_str is None or "{" in path_str:  # missing path or AS-set: skip
        continue
    n_used += 1
    if n_used % 5_000_000 == 0:
        print(f"  {n_used:,} viewpoint routes...")
    vp_total_w[j] += w
    vp_total_u[j] += 1
    for asn in set(path_str.split()):  # a set collapses prepending on its own
        acc = hege_acc.get(asn)
        if acc is None:
            acc = hege_acc[asn] = (array("q", zero_row), array("q", zero_row))
        acc[0][j] += w
        acc[1][j] += 1

gc.enable()
gc.unfreeze()
print(f"{n_used:,} routes from {m} candidate viewpoints, "
      f"{len(hege_acc):,} ASes on-path ({time.time() - t0:.0f}s)")

### Trim, average, rank

The full-feed check is first repeated against what actually survived
deduplication: a peer whose pass-1 announcement count was inflated by feeding
several collectors is dropped here if its distinct usable routes fall short of
the threshold. Then, per AS, the per-viewpoint scores $BC^{(j)}(v)$ — zeros
included: a viewpoint that saw no path through $v$ still votes — are sorted,
the top and bottom $\lfloor\alpha n\rfloor$ discarded, and the rest averaged;
once with address weights ($H_w$) and once unweighted ($H_u$). ASes whose every
surviving score is zero (on-path only for trimmed viewpoints) are dropped.

The `viewpoints` column counts the full-feed viewpoints that saw the AS on at
least one path, before trimming — a low value means the score rests on few
vantage points.

In [ ]:
full = [j for j in range(m) if vp_total_u[j] >= vp_threshold]
n_vp = len(full)
if n_vp == 0:
    raise RuntimeError("no full-feed viewpoints — lower FULL_FEED_FRACTION or add collectors")
k = int(ALPHA * n_vp)
print(f"{n_vp} of {m} candidate viewpoints confirmed full-feed after dedup; "
      f"trimming {k} viewpoint(s) at each end (alpha={ALPHA})")
if n_vp < 20:
    print("WARNING: hegemony is unstable below ~20 viewpoints (paper, Fig. 1b)")

t0 = time.time()
totw = [vp_total_w[j] for j in full]
totu = [vp_total_u[j] for j in full]

hege_rows = []
for asn_s, (aw, au) in hege_acc.items():
    bc_w = sorted(aw[j] / tw for j, tw in zip(full, totw))
    bc_u = sorted(au[j] / tu for j, tu in zip(full, totu))
    hw = sum(bc_w[k:n_vp - k]) / (n_vp - 2 * k)
    hu = sum(bc_u[k:n_vp - k]) / (n_vp - 2 * k)
    if hw == 0 and hu == 0:  # visible only through trimmed viewpoints
        continue
    asn = int(asn_s)
    name, country = asn_info.get(asn, ("", ""))
    hege_rows.append({
        "asn": asn,
        "name": name[:48],
        "country": country,
        "hegemony_w": hw,
        "hegemony_u": hu,
        "viewpoints": sum(1 for x in bc_u if x > 0),
    })

hege_df = (pd.DataFrame(hege_rows)
             .sort_values("hegemony_w", ascending=False)
             .reset_index(drop=True))
hege_df.insert(0, "rank_w", hege_df.index + 1)
print(f"{len(hege_df):,} ASes scored ({time.time() - t0:.0f}s)")

hege_df.head(25).style.format({
    "hegemony_w": "{:.4f}",
    "hegemony_u": "{:.4f}",
}).hide(axis="index")

### Hegemony vs. pooled betweenness centrality

Each point below is one AS. Pure edge ASes sit at $BC = 0$ (hegemony still
scores them — endpoints count — but they cannot render on log axes), so what is
shown is the transit population. On the diagonal the two metrics agree. ASes
**below** the diagonal are ones the pooled metric promotes relative to hegemony
— the signature of vantage-point bias, typically ASes hosting or adjacent to
many collector peers, whose inflated per-viewpoint scores get trimmed or
outvoted here. ASes **above** it are under-sampled in the pooled distinct-route
set but still crossed by a typical viewpoint's table — and hegemony also
credits each AS its own originated address space, which pooled BC never does.

In [ ]:
both = df.merge(hege_df, on="asn", suffixes=("_bc", "_hege"))
pos = both[(both["bc_weighted"] > 0) & (both["hegemony_w"] > 0)]

fig, ax = plt.subplots(figsize=(6.5, 6))
fig.patch.set_facecolor(SURFACE)
style_axes(ax)

lo = max(min(pos["bc_weighted"].min(), pos["hegemony_w"].min()), 1e-9)
hi = max(pos["bc_weighted"].max(), pos["hegemony_w"].max()) * 2
ax.plot([lo, hi], [lo, hi], color=MUTED, linewidth=1, linestyle="--", zorder=1)
ax.annotate("equal under both metrics", xy=(hi, hi),
            xytext=(-8, -14), textcoords="offset points",
            ha="right", fontsize=8, color=MUTED)

ax.scatter(pos["bc_weighted"], pos["hegemony_w"],
           s=14, color=BLUE, alpha=0.45, linewidths=0, zorder=2)

for _, row in pos.nlargest(8, "hegemony_w").iterrows():
    ax.annotate(f"AS{row['asn']}", xy=(row["bc_weighted"], row["hegemony_w"]),
                xytext=(5, 3), textcoords="offset points",
                fontsize=8, color=INK2)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel("Pooled betweenness centrality (address-weighted)", color=INK2)
ax.set_ylabel("AS hegemony (address-weighted)", color=INK2)
ax.set_title(f"AS hegemony vs. pooled BC ({LABEL}, {SNAPSHOT_DATE})", color=INK)
fig.tight_layout()
plt.show()

### Hegemony caveats

- **Viewpoints are peer routers, not peer ASes.** A viewpoint is identified by
  its peer IP, so several routers of the same AS count as several viewpoints —
  as in the paper, whose 326 viewpoints are BGP peers. The trimmed mean absorbs
  moderate redundancy, but a single-collector run draws every viewpoint from one
  collector's peering, and the paper explicitly notes that viewpoints taken from
  a single collector yield poor hegemony estimates. Treat single-collector
  output as a smoke test; aggregate several collectors for real numbers.
- **The full-feed threshold is a heuristic.** The paper says "full-feed BGP
  peers" without a number; requiring `FULL_FEED_FRACTION` (75%) of the union of
  announced IPv4 prefixes is this notebook's operationalization. Partial feeds
  must be excluded — a peer announcing only its own routes would score its
  upstreams at 1.0 and everyone else at 0 — but the exact cutoff barely matters,
  since real tables cluster near the full size.
- **Endpoints count, so hegemony and BC are not per-AS comparable.** An origin
  AS earns hegemony from its own announced address space alone; the pooled BC
  only ever credits transit hops. Comparisons (as in the scatter above) are only
  meaningful on ASes present in both, and even there hegemony includes each
  AS's self-originated mass.
- **Deaggregation is global.** A prefix's weight subtracts more-specifics
  announced by *any* peer, matching the paper's covered-prefix definition, even
  if a particular viewpoint does not carry the more-specific route.
- **Against IHR's published values** (`ihr_hegemony_top10.ipynb`,
  `originasn=0`): same definition and $\alpha$, but IHR draws viewpoints from
  Route Views *and* RIS at 15-minute timebins. Expect the same networks at the
  top of both rankings, not identical scores.
- **Local graphs are the natural extension.** The paper's per-origin hegemony —
  *which networks does this AS depend on* — repeats pass 3 with the routes
  partitioned by origin AS and a trim per (origin, transit) pair; the IHR API's
  non-zero `originasn` values are exactly that.